# Fatigue modeling

Ordinal and classification models with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

Tuning and CV use **train/val participants only**; held-out test participants never appear in Optuna or CV folds.


In [67]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [69]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    add_delta_vs_baseline,
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HIGH_FATIGUE_THRESHOLD,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    STABILITY_SEEDS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import (
    CLASSIFICATION_MODELS,
    HISTORY_ORDINAL_MODELS,
    ORDINAL_MODELS,
    RESIDUAL_ORDINAL_MODELS,
)
from modeling.runner import tune_and_benchmark_model
from modeling.validation import run_stability_study, summarize_stability


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant high-fatigue rate (`prepare_splits(..., stratify=True)`) so train/val and test have similar class balance.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar proportion of people who often report high fatigue — not just a random 8 people who might all happen to be high-fatigue reporters.”


In [70]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)
y_high_fatigue = (df['fatigue_num'] >= HIGH_FATIGUE_THRESHOLD).astype(int)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle, y_high_fatigue))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue,high_fatigue_rate
0,train_val,34,2646,2.419123,0.267196
1,test,8,685,2.816058,0.319708


Test participant ids: [np.int64(10), np.int64(18), np.int64(30), np.int64(37), np.int64(38), np.int64(42), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [71]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
classification_results = []
history_ordinal_results = []
ordinal_best_params = {}
classification_best_params = {}
history_best_params = {}

## 1b. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [60]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, task='ordinal', n_splits=N_CV_FOLDS)
classification_baseline_results = run_all_baseline_benchmarks(bundle, task='classification', n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results, task='ordinal')
clf_baseline_summary = summarize_baseline_metrics(classification_baseline_results, task='classification')

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])
print('Classification baselines (test metrics)')
display(clf_baseline_summary[[c for c in clf_baseline_summary.columns if c.startswith('test_')]])

Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.321168,1.585979,-0.360095,0.000000
global_mode,0.986861,1.372302,-0.018295,0.000000
lag1_fatigue,0.931387,1.358402,0.002229,0.498804
expanding_mean,0.836496,1.162052,0.269827,0.463108


Classification baselines (test metrics)


,test_accuracy,test_f1,test_precision,test_recall
model,,,,
majority_class,0.680292,0.0,0.0,0.0


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 2. Train/Tune models

Run only the model cells you need. Each cell tunes (Optuna) and benchmarks one model. Skip slow models like `lstm` unless required.


### History


Added **History features** (7 cols):
- fatigue lag1: Yesterday’s fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- active_minutes_roll3_mean: Recent typical total active time (lightly + moderately + very)
- calories_sum_roll3_mean: Recent typical daily calories burned
- very_roll3_mean: Recent typical “very active” minutes

#### `catboost_history`


In [61]:
_name = 'catboost_history'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    HISTORY_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_history  test_mae=0.8058
  params={'iterations': 405, 'depth': 4, 'learning_rate': 0.011391140569511004, 'l2_leaf_reg': 5.496839799379424, 'ewma_alpha': 0.2186898857690878, 'rolling_window': 3, 'loss_mode': 'rmse'}
  history_params={'ewma_alpha': 0.2186898857690878, 'rolling_window': 3}


**`catboost_history`** jointly tunes EWMA alpha, rolling window, CatBoost params, and loss mode (`rmse` vs `multiclass`).
- RMSE: regression with clip for out-of-boundary predictions
- Multiclass: classification

#### `catboost_residual_expanding`


In [ ]:
_name = 'catboost_residual_expanding'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    RESIDUAL_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


**`catboost_residual_expanding`**: `pred = clip(expanding_mean + CatBoost_residual)`. Jointly tunes EWMA alpha, rolling window, and CatBoost params (RMSE on residual only).

### Ordinal

#### `ordered_logistic`


In [48]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.2730


#### `ordinal_rf`


In [49]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.0467


#### `catboost_ordinal`


In [50]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.2088


#### `mixed_effects`


In [51]:
_name = 'mixed_effects'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] mixed_effects  test_mae=1.2117


#### `lstm` (slow)

In [ ]:
# _name = 'lstm'
# _result, _params = tune_and_benchmark_model(
#     _name,
#     bundle,
#     ORDINAL_MODELS,
#     task='ordinal',
#     n_trials=OPTUNA_TRIALS,
#     n_splits=N_CV_FOLDS,
# )
# ordinal_results.append(_result)
# ordinal_best_params[_name] = _params
# print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')

### Classification

#### `lightgbm`


In [ ]:
_name = 'lightgbm'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')


#### `random_forest`


In [ ]:
_name = 'random_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    CLASSIFICATION_MODELS,
    task='classification',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
classification_results.append(_result)
classification_best_params[_name] = _params
print(f'[ok] {_name}  test_f1={_result["test_metrics"]["f1"]:.4f}')


[ok] random_forest  test_f1=0.4386


## 4. Results summary

Includes baselines plus only models whose §2 cells were executed. Ordinal CV summary shows **`cv_mae_std` only** (fold stability for MAE). Test summary includes **delta vs `expanding_mean`** and **delta vs `lag1_fatigue`**.

In [62]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    metric_cols = ['mae', 'rmse', 'r2', 'qwk'] if task == 'ordinal' else ['accuracy', 'f1', 'precision', 'recall']
    for result in results:
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        test_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            test_row[f'test_{col}'] = result['test_metrics'][col]

        if task == 'ordinal':
            cv_row['cv_mae_std'] = cv_std['mae']

        cv_rows.append(cv_row)
        test_rows.append(test_row)
    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
classification_results = globals().get('classification_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})
classification_best_params = globals().get('classification_best_params', {})

ran_ordinal = sorted(set(ordinal_best_params) | set(history_best_params))
ran_classification = sorted(classification_best_params.keys())
print(f'Ran {len(ran_ordinal)} tuned ordinal models: {ran_ordinal}')
print(f'Ran {len(ran_classification)} tuned classification models: {ran_classification}')

all_ordinal_results = ordinal_baseline_results + ordinal_results + history_ordinal_results
all_classification_results = classification_baseline_results + classification_results

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results, task='ordinal')
clf_cv_summary, clf_test_summary = collect_summaries(all_classification_results, task='classification')

ordinal_test_summary = add_delta_vs_baseline(
    ordinal_test_summary,
    baseline_name='expanding_mean',
    metric_cols=['mae', 'rmse', 'r2', 'qwk'],
    task='ordinal',
)
ordinal_test_summary = add_delta_vs_baseline(
    ordinal_test_summary,
    baseline_name='lag1_fatigue',
    metric_cols=['mae', 'rmse', 'r2', 'qwk'],
    task='ordinal',
)
clf_test_summary = add_delta_vs_baseline(
    clf_test_summary,
    baseline_name='majority_class',
    metric_cols=['accuracy', 'f1', 'precision', 'recall'],
    task='classification',
)

print('Ordinal CV summary (baselines first)')
display(ordinal_cv_summary)
print('Ordinal held-out test summary (delta vs expanding_mean and lag1_fatigue)')
display(ordinal_test_summary)
print('Classification CV summary (baselines first)')
display(clf_cv_summary)
print('Classification held-out test summary (delta vs majority_class)')
display(clf_test_summary)


Ran 1 tuned ordinal models: ['catboost_history']
Ran 0 tuned classification models: []
Ordinal CV summary (baselines first)


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
global_mean,{},1.317778,1.552164,-0.088215,0.000000,0.057900
global_mode,{},1.257337,1.601647,-0.157108,0.000000,0.147267
lag1_fatigue,{},0.829233,1.339758,0.179388,0.587082,0.132826
expanding_mean,{},0.916134,1.246354,0.298481,0.519738,0.111049
catboost_history,"{'iterations': 405, 'depth': 4, 'learning_rate...",0.870913,1.177423,0.372019,0.540724,0.095477


Ordinal held-out test summary (delta vs expanding_mean and lag1_fatigue)


,best_params,test_mae,test_rmse,test_r2,test_qwk,delta_mae_vs_expanding_mean,delta_rmse_vs_expanding_mean,delta_r2_vs_expanding_mean,delta_qwk_vs_expanding_mean,delta_mae_vs_lag1_fatigue,delta_rmse_vs_lag1_fatigue,delta_r2_vs_lag1_fatigue,delta_qwk_vs_lag1_fatigue
model,,,,,,,,,,,,,
global_mean,{},1.321168,1.585979,-0.360095,0.000000,0.484672,0.423927,-0.629922,-0.463108,0.389781,0.227577,-0.362324,-0.498804
global_mode,{},0.986861,1.372302,-0.018295,0.000000,0.150365,0.210250,-0.288122,-0.463108,0.055474,0.013900,-0.020524,-0.498804
lag1_fatigue,{},0.931387,1.358402,0.002229,0.498804,0.094891,0.196350,-0.267598,0.035696,0.000000,0.000000,0.000000,0.000000
expanding_mean,{},0.836496,1.162052,0.269827,0.463108,0.000000,0.000000,0.000000,0.000000,-0.094891,-0.196350,0.267598,-0.035696
catboost_history,"{'iterations': 405, 'depth': 4, 'learning_rate...",0.805839,1.125679,0.314822,0.496471,-0.030657,-0.036373,0.044994,0.033363,-0.125547,-0.232723,0.312593,-0.002333


Classification CV summary (baselines first)


,best_params,cv_accuracy,cv_f1,cv_precision,cv_recall
model,,,,,
majority_class,{},0.732127,0.0,0.0,0.0


Classification held-out test summary (delta vs majority_class)


,best_params,test_accuracy,test_f1,test_precision,test_recall,delta_accuracy_vs_majority_class,delta_f1_vs_majority_class,delta_precision_vs_majority_class,delta_recall_vs_majority_class
model,,,,,,,,,
majority_class,{},0.680292,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5 Model Stability

Run each cell separately. Each tuned model is re-fit with full Optuna on every seed. The summary cell adds `expanding_mean` (no tuning) and compares all three. Use `_stability_seeds = [42, 43, 44]` in the first cell for a quick smoke test.

Negative `delta_mae_vs_expanding_mean` or `delta_mae_vs_catboost_history` in the summary means improvement.


In [ ]:
# Optional: use fewer seeds for a quick run
# _stability_seeds = [42, 43, 44]
_stability_seeds = STABILITY_SEEDS
stability_parts = globals().get('stability_parts', {})

_stability_history = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_history'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_history'] = _stability_history
print(f'[ok] catboost_history stability  rows={len(_stability_history)}')
display(_stability_history)


In [ ]:
stability_parts = globals().get('stability_parts', {})

_stability_residual = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_residual_expanding'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_residual_expanding'] = _stability_residual
print(f'[ok] catboost_residual_expanding stability  rows={len(_stability_residual)}')
display(_stability_residual)


In [ ]:
stability_parts = globals().get('stability_parts', {})
_seeds = globals().get('_stability_seeds', STABILITY_SEEDS)

_stability_baseline = run_stability_study(
    df,
    seeds=_seeds,
    models=['expanding_mean'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)

parts = [_stability_baseline]
for _name in ['catboost_history', 'catboost_residual_expanding']:
    if _name in stability_parts:
        parts.append(stability_parts[_name])

stability_df = pd.concat(parts, ignore_index=True)
stability_summary = summarize_stability(stability_df)

print('Combined per-seed stability results')
display(stability_df)
print('Aggregated summary (negative delta vs expanding_mean or catboost_history = improvement)')
display(stability_summary)
